<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/debug_hct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!pip install -r cursivetransformer/requirements.txt

In [ ]:
import sys
sys.path.append('/content/cursivetransformer')
from cursivetransformer.model import get_all_args, get_checkpoint, get_latest_checkpoint_artifact
from cursivetransformer.data import create_datasets, offsets_to_strokes, strokes_to_offsets
from cursivetransformer.sample import generate, generate_n_words, plot_strokes
from cursivetransformer.mech_interp import (
    HookedCursiveTransformer,
    HookedCursiveTransformerConfig,
    convert_cursivetransformer_model_config,
    visualize_attention,
    generate_repeated_stroke_tokens,
    generate_random_ascii_context,
    run_and_cache_model_repeated_tokens,
    compute_induction_scores,
    plot_induction_scores,
    plot_head_attention_pattern,
    create_induction_summary,
    ablate_heads,
    get_induction_positions,
    compute_loss_on_induction_positions,
    visualize_attention_patterns
)

import pandas as pd
import os

import copy
import types
from typing import List, Callable, Dict, Optional, Union, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import circuitsvis as cv
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from jaxtyping import Float, Int


import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import ActivationCache

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# wandb_api_key - e56bbe426145e5983e72a933299daca195ebb6a7
args = get_all_args(False)
args.sample_only = True
args.max_seq_length = 1250
args.load_from_run_id = '7e9hz1og'
args.wandb_project = 'bigbank_2k'
args.wandb_entity = 'sam-greydanus'
args.dataset_name = 'bigbank'
args.wandb_run_name = 'cursivetransformer_mech_interp_part_3_stroke_attention'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.vocab_size = train_dataset.get_vocab_size()
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.context_vocab_size = train_dataset.get_char_vocab_size()

## Verify that the original and hooked versions of the model produce the same output logits for a given sequence

In [ ]:
# Load the original model
original_model, _, _, _, _ = get_checkpoint(args, sample_only=True)
original_model.eval()

In [ ]:
# Load the hooked model
hooked_model = HookedCursiveTransformer.from_pretrained(args)
hooked_model = hooked_model.to(device)
hooked_model.eval()

In [ ]:
# Prepare a sample input
sample_tokens = torch.randint(0, args.vocab_size, (1, args.block_size)).to(device)
sample_context = torch.randint(0, args.context_vocab_size, (1, args.context_block_size)).to(device)

In [ ]:
# Run both models
with torch.no_grad():
    original_output = original_model(sample_tokens, sample_context)
    hooked_output = hooked_model(sample_tokens, sample_context)

In [ ]:
# Compare outputs
max_diff = torch.max(torch.abs(original_output[0] - hooked_output))
print(f"Maximum difference in logits: {max_diff.item()}")

if max_diff > 1e-5:
    print("Models are not equivalent!")
    # If models are not equivalent, let's print more detailed information
    print("Original model output shape:", original_output[0].shape)
    print("Hooked model output shape:", hooked_output.shape)
    print("Original model output mean:", original_output[0].mean().item())
    print("Hooked model output mean:", hooked_output.mean().item())
    print("Original model output std:", original_output[0].std().item())
    print("Hooked model output std:", hooked_output.std().item())
else:
    print("Models are equivalent.")

# Scratch

In [ ]:
import einops
import torch

from transformer_lens.HookedTransformerConfig import HookedTransformerConfig


# - [ ] TODO: Add cross-attention
def convert_gpt2_weights(cursivetransformer, cfg: HookedTransformerConfig):
    state_dict = {}

    state_dict["embed.W_E"] = cursivetransformer.transformer.wte.weight
    state_dict["pos_embed.W_pos"] = cursivetransformer.transformer.wpe.weight

    for l in range(cfg.n_layers):
        state_dict[f"blocks.{l}.ln1.w"] = cursivetransformer.transformer.h[l].ln_1.weight
        state_dict[f"blocks.{l}.ln1.b"] = cursivetransformer.transformer.h[l].ln_1.bias

        # In GPT-2, q,k,v are produced by one big linear map, whose output is
        # concat([q, k, v])
        W = cursivetransformer.transformer.h[l].attn.c_attn.weight
        W_Q, W_K, W_V = torch.tensor_split(W, 3, dim=1)
        W_Q = einops.rearrange(W_Q, "m (i h)->i m h", i=cfg.n_heads)
        W_K = einops.rearrange(W_K, "m (i h)->i m h", i=cfg.n_heads)
        W_V = einops.rearrange(W_V, "m (i h)->i m h", i=cfg.n_heads)

        state_dict[f"blocks.{l}.attn.W_Q"] = W_Q
        state_dict[f"blocks.{l}.attn.W_K"] = W_K
        state_dict[f"blocks.{l}.attn.W_V"] = W_V

        qkv_bias = cursivetransformer.transformer.h[l].attn.c_attn.bias
        qkv_bias = einops.rearrange(
            qkv_bias,
            "(qkv index head)->qkv index head",
            qkv=3,
            index=cfg.n_heads,
            head=cfg.d_head,
        )
        state_dict[f"blocks.{l}.attn.b_Q"] = qkv_bias[0]
        state_dict[f"blocks.{l}.attn.b_K"] = qkv_bias[1]
        state_dict[f"blocks.{l}.attn.b_V"] = qkv_bias[2]

        W_O = cursivetransformer.transformer.h[l].attn.c_proj.weight
        W_O = einops.rearrange(W_O, "(i h) m->i h m", i=cfg.n_heads)
        state_dict[f"blocks.{l}.attn.W_O"] = W_O
        state_dict[f"blocks.{l}.attn.b_O"] = cursivetransformer.transformer.h[l].attn.c_proj.bias

        state_dict[f"blocks.{l}.ln2.w"] = cursivetransformer.transformer.h[l].ln_2.weight
        state_dict[f"blocks.{l}.ln2.b"] = cursivetransformer.transformer.h[l].ln_2.bias

        W_in = cursivetransformer.transformer.h[l].mlp.c_fc.weight
        state_dict[f"blocks.{l}.mlp.W_in"] = W_in
        state_dict[f"blocks.{l}.mlp.b_in"] = cursivetransformer.transformer.h[l].mlp.c_fc.bias

        W_out = cursivetransformer.transformer.h[l].mlp.c_proj.weight
        state_dict[f"blocks.{l}.mlp.W_out"] = W_out
        state_dict[f"blocks.{l}.mlp.b_out"] = cursivetransformer.transformer.h[l].mlp.c_proj.bias
    state_dict["unembed.W_U"] = cursivetransformer.lm_head.weight.T

    state_dict["ln_final.w"] = cursivetransformer.transformer.ln_f.weight
    state_dict["ln_final.b"] = cursivetransformer.transformer.ln_f.bias
    return state_dict